# 003 · Key-Value Storage with shelve and dbm

> **Standard Library › Persistence** · Python 3.14

Sometimes you don't need tables or SQL — you just want a dictionary that survives a restart. `dbm` is a simple on-disk key-value store for bytes, and `shelve` builds on it to give you a **persistent `dict`** that can hold almost any Python object.

**In this notebook you will learn:**

- `dbm`: an on-disk store of `bytes` keys and values, and its backends (including `dbm.sqlite3`, the default since 3.13)
- `shelve`: a persistent dict of pickled objects
- the `writeback` gotcha when mutating stored objects
- the security caveat (shelve is pickle underneath)
- when to choose SQLite or JSON instead

## .NET mental model

There is no built-in persistent dictionary in the .NET BCL. The closest things are:

| .NET | Python |
|---|---|
| a `Dictionary<string, byte[]>` you serialise to a file yourself | `dbm` (bytes → bytes, on disk) |
| an embedded store from NuGet (e.g. LiteDB) | `dbm` / `shelve` |
| `Dictionary<string, object>` + `BinaryFormatter` to disk (insecure, removed) | `shelve` (pickle-backed — same caveat!) |

All files in this notebook live in a `TemporaryDirectory` *(see Standard Library › Files & Paths › 004 · Temporary Files and Directories)*.

## `dbm`: a persistent bytes-to-bytes map

`dbm.open(path, flag)` returns a dict-like object. Keys and values are stored as **bytes**; `str` is accepted and encoded as UTF-8, but you always get `bytes` back.

| Flag | Meaning |
|---|---|
| `"r"` | read-only, must exist (default) |
| `"w"` | read/write, must exist |
| `"c"` | read/write, create if missing |
| `"n"` | always create a new, empty database |

In [ ]:
import dbm
import tempfile
from pathlib import Path

tmp = tempfile.TemporaryDirectory()        # cleaned up at the end of the notebook
root = Path(tmp.name)

with dbm.open(root / "cache", "c") as db:
    db["greeting"] = "hello"               # str is encoded to UTF-8
    db[b"user:42"] = b"alice"
    print(db["greeting"])                  # b'hello' — bytes back!
    print(db.get("missing"), len(db))
    print("user:42" in db, sorted(db.keys()))

The data survives closing and reopening:

In [ ]:
with dbm.open(root / "cache", "r") as db:
    print(db["user:42"].decode())

⚠️ **Gotcha:** values are bytes. Store structured data by encoding it yourself — JSON is a good fit — and don't rely on other types: what happens with an `int` or a `list` depends on the backend (some convert, some raise).

In [ ]:
import json

with dbm.open(root / "cache", "w") as db:
    db["settings"] = json.dumps({"theme": "dark", "size": 14})
    print(json.loads(db["settings"]))

    try:
        db["tags"] = ["a", "b"]            # not bytes or str
    except Exception as e:
        print(f"{type(e).__name__}: {e}")

Opening a database that doesn't exist with `"r"` or `"w"` raises `dbm.error` — which is actually a *tuple* of exception classes, so it works in an `except` clause:

In [ ]:
try:
    dbm.open(root / "no-such-db", "r")
except dbm.error as e:
    print(f"{type(e).__name__}: {e}")

## `dbm` backends

`dbm` is a front end for several implementations. `dbm.open` picks the first available when **creating** a database, and detects the format when opening an existing one (`dbm.whichdb`).

| Module | Notes |
|---|---|
| `dbm.sqlite3` | new in **3.13** and the default when available — a single SQLite file, portable |
| `dbm.gnu` | GNU gdbm — only if Python was built with it |
| `dbm.ndbm` | Unix ndbm/Berkeley DB — platform-dependent |
| `dbm.dumb` | pure-Python fallback, always available, slow; creates `.dat`/`.dir`/`.bak` files |

Older backends differ in file names and portability, which is one reason `dbm` files traditionally weren't moved between machines. With `dbm.sqlite3` that's much less of a problem.

In [ ]:
print(dbm.whichdb(root / "cache"))
print(sorted(p.name for p in root.iterdir()))

import dbm.dumb
with dbm.dumb.open(str(root / "legacy"), "c") as db:   # force a specific backend
    db["k"] = "v"
print(sorted(p.name for p in root.iterdir()))

## `shelve`: a persistent dict of Python objects

`shelve.open(path)` gives you a dict-like **shelf** whose keys are `str` and whose values can be any picklable object *(see Standard Library › Data Formats › 005 · Serialisation with pickle)*. Under the hood it pickles each value and stores it in a `dbm` database.

In [ ]:
import shelve
from dataclasses import dataclass
from datetime import datetime

@dataclass
class Session:
    user: str
    started: datetime
    cart: list[str]

with shelve.open(root / "sessions") as shelf:
    shelf["s1"] = Session("alice", datetime(2025, 3, 1, 9, 0), ["apple"])
    shelf["s2"] = Session("bob", datetime(2025, 3, 1, 9, 5), [])
    shelf["counters"] = {"visits": 2}

with shelve.open(root / "sessions") as shelf:          # reopen: everything is back
    print(sorted(shelf.keys()))
    print(shelf["s1"])
    print(shelf.get("s9", "no such session"))

It supports the usual dict operations — `in`, `len`, `del`, iteration, `get`, `setdefault`, `update`. Keys must be strings:

In [ ]:
with shelve.open(root / "sessions") as shelf:
    del shelf["s2"]
    print("s2" in shelf, len(shelf))
    try:
        shelf[42] = "int key"
    except (TypeError, AttributeError) as e:
        print(f"{type(e).__name__}: {e}")

## ⚠️ Gotcha: mutating a stored object does nothing

`shelf["s1"]` **unpickles a fresh copy** every time you read it. Mutating that copy changes nothing on disk — the classic shelve bug:

In [ ]:
with shelve.open(root / "sessions") as shelf:
    shelf["s1"].cart.append("pear")          # modifies a temporary copy
    print(shelf["s1"].cart)                  # ['apple'] — the change is lost!

Two fixes:

1. **Read, modify, write back** (explicit — recommended):

In [ ]:
with shelve.open(root / "sessions") as shelf:
    session = shelf["s1"]
    session.cart.append("pear")
    shelf["s1"] = session                    # store it again
    print(shelf["s1"].cart)

2. **`writeback=True`**: the shelf caches every entry you access and writes them all back on `sync()` / `close()`. Convenient, but it uses memory for every accessed entry and makes closing slow on large shelves.

In [ ]:
with shelve.open(root / "sessions", writeback=True) as shelf:
    shelf["s1"].cart.append("melon")         # mutated in the cache
    shelf["counters"]["visits"] += 1
    # written back when the with-block closes the shelf

with shelve.open(root / "sessions") as shelf:
    print(shelf["s1"].cart, shelf["counters"])

## ⚠️ Security: shelve is pickle

Because every value is a pickle, opening a shelf file from an untrusted source can **execute arbitrary code** — exactly like `pickle.loads` (and like the removed `BinaryFormatter`). Only open shelves that your own program created. Also, since pickles store classes by reference, renaming or moving a class (like `Session`) breaks existing shelves.

## Concurrency and robustness

- Neither `shelve` nor `dbm` supports **concurrent writers** in general (`dbm.sqlite3` inherits SQLite's locking, but `shelve` itself has no transaction or conflict handling).
- There are no queries, indexes or schema — only lookups by key.
- Always close the shelf (use `with`), or recent writes may not be flushed.

## Choosing a storage option

| Need | Use |
|---|---|
| Quick persistent cache of Python objects for one program | `shelve` |
| Simple string/bytes key-value store | `dbm` (or a SQLite table) |
| Human-readable config or small data shared with other tools | JSON / TOML files |
| Queries, multiple tables, transactions, concurrent readers | `sqlite3` *(see Standard Library › Persistence › 001 · SQLite Basics)* |
| Data from untrusted sources | JSON or SQLite — never `shelve`/`pickle` |

In practice, many Python developers skip `shelve` and go straight to `sqlite3` + JSON columns: it is just as easy, safe to share, inspectable with standard tools, and supports transactions. A tiny key-value store on SQLite is only a few lines:

In [ ]:
import sqlite3
from contextlib import closing

with closing(sqlite3.connect(root / "kv.db")) as conn, conn:
    conn.execute("CREATE TABLE IF NOT EXISTS kv (key TEXT PRIMARY KEY, value TEXT NOT NULL)")
    conn.execute("INSERT OR REPLACE INTO kv VALUES (?, ?)", ("s1", json.dumps({"user": "alice", "cart": ["apple"]})))
    value = conn.execute("SELECT value FROM kv WHERE key = ?", ("s1",)).fetchone()[0]
    print(json.loads(value))

In [ ]:
tmp.cleanup()                                # remove all files created in this notebook
print(root.exists())

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Open/create a key-value file | (NuGet store, or roll your own) | `dbm.open(path, "c")` |
| Store bytes | `dict[key] = bytes` + save | `db[key] = b"..."` |
| Persistent dict of objects | `Dictionary<string, object>` + serializer | `shelve.open(path)` |
| Update a stored object | read, modify, save | read, modify, `shelf[key] = obj` (or `writeback=True`) |
| Which format is this file? | — | `dbm.whichdb(path)` |
| Untrusted data | `System.Text.Json` | JSON / SQLite — never `shelve` |

## Exercises

1. Use `dbm` to count word occurrences in `"the cat and the hat and the bat"` (values stored as decimal strings), then reopen the database read-only and print the counts sorted by word.
2. Store a `dict` of `{"alice": ["admin"], "bob": []}` in a shelf under the key `"roles"`. Then add `"editor"` to bob's roles **without** `writeback=True`, and verify the change persisted after reopening.
3. Explain why the following code prints `[]` and fix it with `writeback=True`:
   ```python
   with shelve.open(path) as s:
       s["queue"] = []
       s["queue"].append("job-1")
       print(s["queue"])
   ```

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import dbm
import shelve
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)

    # 1 — encode/decode the counts yourself
    with dbm.open(root / "words", "c") as db:
        for word in "the cat and the hat and the bat".split():
            db[word] = str(int(db.get(word, b"0")) + 1)
    with dbm.open(root / "words", "r") as db:
        for key in sorted(db.keys()):
            print(key.decode(), int(db[key]))

    # 2 — read, modify, write back
    with shelve.open(root / "acl") as s:
        s["roles"] = {"alice": ["admin"], "bob": []}
    with shelve.open(root / "acl") as s:
        roles = s["roles"]
        roles["bob"].append("editor")
        s["roles"] = roles
    with shelve.open(root / "acl") as s:
        print(s["roles"])   # {'alice': ['admin'], 'bob': ['editor']}

    # 3 — each s["queue"] read unpickles a NEW list, so append() modified a throwaway copy.
    #     With writeback=True the shelf caches the object and the append sticks.
    with shelve.open(root / "jobs", writeback=True) as s:
        s["queue"] = []
        s["queue"].append("job-1")
        print(s["queue"])   # ['job-1']
```

</details>